# Overnight pipeline on Kaggle

Runs the long jobs that don't fit on the local laptop overnight:

1. Full-set BLIP-2 ITM re-rank evaluation on condition `C_alpha0.7_hn` (~3–5 h on T4/P100)
2. Two extra hard-neg fine-tuning runs with seeds 588 and 527 (~1 h each on T4)
3. Rebuild C-condition HNSW indices and evaluate per seed

## Attach these datasets before running

- `ashok1145/vr-fproj`               — DeepFashion images + annotations
- `taralsanka/blip-updated-captions` — `captions.json` (BLIP-2)
- `taralsanka/best-yolo-pt`          — YOLO weights
- `taralsanka/clip-saved-outputs`    — Vanilla fine-tuned CLIP + 5 original HNSW indices
- **A dataset containing `clip_finetuned_hn.pt`** that you uploaded — slug must be one of:
  - `ashok1145/clip-hn-checkpoint`
  - `ashok1145/clip-finetuned-hn`
  - or set the env var `HN_CKPT_PATH` to its file path in the cell below.

Enable **Internet** and select **GPU (T4 ×2, P100, or T4 ×1)** in the notebook sidebar.

In [ ]:
# ── 0. Clone code + install deps ──────────────────────────────────────────
GITHUB_REPO = "ashokCh-dev/VR_Final_project"
BRANCH = "main"

import os
os.chdir('/kaggle/working')
if not os.path.isdir('VR_final_proj'):
    !git clone -b {BRANCH} https://github.com/{GITHUB_REPO}.git VR_final_proj
%cd /kaggle/working/VR_final_proj
!pip install --quiet open_clip_torch hnswlib bitsandbytes accelerate 2>&1 | tail -3

In [ ]:
# ── 1. Verify environment + paths ─────────────────────────────────────────
import sys, torch
sys.path.insert(0, '/kaggle/working/VR_final_proj')
from src import config
print('cuda available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU             :', torch.cuda.get_device_name(0))
    print('VRAM            :', f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('IS_KAGGLE       :', config.IS_KAGGLE)
print('DATASET_ROOT    :', config.DATASET_ROOT, 'exists', config.DATASET_ROOT.exists())
print('IMG_ROOT        :', config.IMG_ROOT, 'exists', config.IMG_ROOT.exists())
print('CAPTIONS_FILE   :', config.CAPTIONS_FILE, 'exists', config.CAPTIONS_FILE.exists())
print('CLIP_FT_HN      :', config.CLIP_FT_HN, 'exists', config.CLIP_FT_HN.exists())
print('YOLO_WEIGHTS    :', config.YOLO_WEIGHTS, 'exists', config.YOLO_WEIGHTS.exists())
print('CLIP_FT (vanilla):', config.CLIP_FT, 'exists', config.CLIP_FT.exists())
print('Vanilla A index :', config.resolve_artifact('gallery_index_A.bin'))
assert config.DATASET_ROOT.exists(), 'Missing vr-fproj dataset — attach it via Add data.'
assert config.CAPTIONS_FILE.exists(), 'Missing blip-updated-captions dataset.'
assert config.CLIP_FT_HN.exists(), 'Missing clip_finetuned_hn.pt dataset. Set HN_CKPT_PATH env var if you uploaded under a different slug.'
print('\nAll paths resolved ✓')

In [ ]:
# ── 1.5. Rebuild C_alpha0.7_hn / C_alpha0.5_hn HNSW indices ─────────────
# The uploaded clip-hn-checkpoint dataset only has the .pt, not the HNSW
# indices built from it — so we rebuild them on Kaggle (~5–10 min, T4).
!python -u rebuild_indices_hn.py 2>&1 | grep -vE "%\|" | tail -10
!ls -la /kaggle/working/artifacts/*.bin /kaggle/working/artifacts/*.json 2>&1 | tail -10

In [ ]:
# ── 2. Sanity baseline: reproduce C_alpha0.7_hn metrics on a small sample ─
!python -u demo_batch_eval.py --condition C_alpha0.7_hn --max_queries 200 \
    --out /kaggle/working/artifacts/sanity_C_hn07.json 2>&1 | grep -vE "%\|" | tail -20

In [ ]:
# ── 3. Full-set BLIP-2 ITM rerank on C_alpha0.7_hn (longest single job) ───
# ~3–5 hours on T4/P100. Saves to /kaggle/working/artifacts/eval_C_hn_itm_blend02_full.json
!python -u demo_batch_eval.py \
    --condition C_alpha0.7_hn \
    --rerank --rerank_mode blend --itm_weight 0.2 \
    --out /kaggle/working/artifacts/eval_C_hn_itm_blend02_full.json \
    2>&1 | grep -vE "%\|" | tail -25

In [ ]:
# ── 4. Multi-seed retraining: seed 588 ────────────────────────────────────
!python -u train_hn.py --seed 588 --suffix _seed588 2>&1 | grep -vE "%\|" | tail -30

In [ ]:
# ── 5. Rebuild + eval seed 588 ────────────────────────────────────────────
!python -u rebuild_indices_hn.py --suffix _seed588 2>&1 | grep -vE "%\|" | tail -15
!python -u demo_batch_eval.py \
    --condition C_alpha0.7_hn_seed588 \
    --bin_path  /kaggle/working/artifacts/gallery_index_C_alpha07_hn_seed588.bin \
    --meta_path /kaggle/working/artifacts/gallery_meta_C_alpha07_hn_seed588.json \
    --checkpoint /kaggle/working/artifacts/clip_finetuned_hn_seed588.pt \
    --alpha 0.7 --model_type ft_hn \
    --out /kaggle/working/artifacts/eval_C_hn_seed588.json \
    2>&1 | grep -vE "%\|" | tail -20

In [ ]:
# ── 6. Multi-seed retraining: seed 527 ────────────────────────────────────
!python -u train_hn.py --seed 527 --suffix _seed527 2>&1 | grep -vE "%\|" | tail -30

In [ ]:
# ── 7. Rebuild + eval seed 527 ────────────────────────────────────────────
!python -u rebuild_indices_hn.py --suffix _seed527 2>&1 | grep -vE "%\|" | tail -15
!python -u demo_batch_eval.py \
    --condition C_alpha0.7_hn_seed527 \
    --bin_path  /kaggle/working/artifacts/gallery_index_C_alpha07_hn_seed527.bin \
    --meta_path /kaggle/working/artifacts/gallery_meta_C_alpha07_hn_seed527.json \
    --checkpoint /kaggle/working/artifacts/clip_finetuned_hn_seed527.pt \
    --alpha 0.7 --model_type ft_hn \
    --out /kaggle/working/artifacts/eval_C_hn_seed527.json \
    2>&1 | grep -vE "%\|" | tail -20

In [ ]:
# ── 8. Summary: dump all numerical results ───────────────────────────────
import json, glob
for p in sorted(glob.glob('/kaggle/working/artifacts/eval_*.json')):
    print(f'\n=== {p.split("/")[-1]} ===')
    payload = json.load(open(p))
    print(f"alpha={payload.get('alpha')}, rerank={payload.get('rerank')}, n={payload.get('n_queries')}")
    for m, v in payload['results'].items():
        print(f'  {m}: {v:.4f}')